*© 2026 Paul Fergus. Free for student and research use — commercial use is strictly prohibited.*

# Lab 6 — Annotating images for object detection

**Module:** Deep Learning Concepts and Techniques (Computer Vision)
**Week:** 6
**Estimated time:** self-paced take-home — you have **one week** before Lab 7. Realistically 15-20+ hours of hands-on annotation across multiple sessions; do not attempt this in one sitting.

---

## Learning outcomes

By the end of this lab you should be able to:

1. Explain the difference between **image classification** (what's in this image?) and **object detection** (what's in this image, where, and how many of each?).
2. Read, write, and *validate* labels in the **YOLO text format**, including normalised centre-based bounding box coordinates.
3. Annotate a folder of images using the module's built-in bounding-box annotator and produce a clean YOLO-format label set.
4. Build a valid Ultralytics `data.yaml` specification linking your image folder, label folder, class list, and train/val/test split.
5. Visualise existing annotations by rendering bounding boxes back onto their source images — the critical 'closing-the-loop' step that catches off-by-one errors and class-mapping mistakes.
6. Apply common annotation hygiene rules — tight boxes, consistent class assignment, handling of edge cases like truncation, occlusion, and ambiguous identification.
7. Explain why real-world datasets usually arrive **pre-split**, and why reshuffling a published split yourself is usually the wrong instinct.

## Prerequisites

- **Labs 1–5** completed. The pipeline knowledge from Lab 5 will be directly extended here.
- The textbook *Applied Deep Learning* (Fergus & Chalmers), Chapter 7 — object detection.
- Lecture 6: *From classification to detection — bounding boxes, IoU, and the YOLO family of models*.

## ⚠ Before you start — add the image pool

This repository doesn't bundle the photo pool itself (size and licensing) — your instructor provides it separately. **Place the `.jpg` files flat into `data/images/` before continuing** (see `data/images/README.md` for details). Everything below assumes that pool is already in place.

## A different kind of lab

Up to now every lab has been about *training a model*. This one is about **preparing the data a model will train on**. That sounds less glamorous than the modelling work, but in industry the ratio is typically 70/30 data work to modelling work, sometimes much higher. **The quality ceiling on any deep learning project is set by the quality of the labels.** A model trained on inconsistent or careless annotations will be inconsistent or careless. There is no clever architecture that fixes bad data.

Today we annotate — and this time, we annotate **everything**. Next week we'll train an object detector on the dataset you produce here.

## The dataset — real photographs, not synthetic ones

This lab uses a large pool of **real photographs** — not synthetic renders — of wildlife across four classes: **buffalo, elephant, rhino, zebra**. Every image lives, unlabelled, in `data/images/`. There is no separate "real" dataset waiting in the wings and no shortcut around doing the work: the annotations you produce this week **are** the dataset Lab 7 trains on, in full. Nobody has pre-split it, pre-annotated it, or verified it for you — that's now entirely your job, at a scale that starts to resemble what annotation work actually looks like outside a classroom.

The images span a realistic range of difficulty: single-animal shots, herd shots with a dozen animals in one frame, and a few genuinely small, distant subjects.

## What you'll build

By the end of this week you will have:

- Every image in `data/images/` annotated with bounding boxes for every animal visible from the four-class list: **buffalo, elephant, rhino, zebra**.
- A matching folder of YOLO-format `.txt` files, one per image.
- A `data.yaml` declaration file that the `ultralytics` training pipeline can ingest.
- A dataset that is entirely your own annotation work, large enough for Lab 7 to train a genuinely useful detector on.

## Useful references

- [Ultralytics dataset format documentation](https://docs.ultralytics.com/datasets/detect/)
- [Roboflow's guide to writing good bounding boxes](https://blog.roboflow.com/labeling/) — vendor blog but covers the hygiene rules well.

---

## 1. The YOLO text format, dissected

Every object detector needs to know *where* objects are, not just *what* they are. The YOLO family encodes a bounding box as four numbers, plus a class id, all on one line:

```
class_id  cx  cy  w  h
```

For example:

```
1 0.523000 0.487000 0.214000 0.299000
```

Reading this off:

| Field      | Meaning |
|------------|---------|
| `1`        | Class id (in our dataset: `0=buffalo, 1=elephant, 2=rhino, 3=zebra`) |
| `0.523000` | **`cx`** — x-coordinate of box centre, **normalised** (0 = left edge, 1 = right edge) |
| `0.487000` | **`cy`** — y-coordinate of box centre, **normalised** (0 = top, 1 = bottom) |
| `0.214000` | **`w`** — box width, **normalised** (fraction of image width) |
| `0.299000` | **`h`** — box height, **normalised** (fraction of image height) |

**Two things worth pausing on:**

1. The format uses the box **centre**, not the corner. Many other formats (Pascal VOC, COCO) use `xmin, ymin, xmax, ymax` corners. Mixing them up is a classic bug — see Exercise 1.
2. All four geometric fields are **normalised by image dimensions**. This means the same `.txt` file is valid whether the image is 800×600 or 4096×3072 — the label is intrinsically resolution-independent. This is why YOLO's format scales so well across heterogeneous datasets, and it's exactly why the images in this dataset can vary in size (they do) without causing you any problems.

**Per-image rule:** one `.txt` file per image, named with the same stem (e.g. `img_0142.jpg` → `img_0142.txt`). One row per object instance. If there are no objects in the image, the `.txt` file is empty (but should still exist).

## 2. The hands-on bit — open the annotator

The module's container ships with a built-in bounding-box annotator that writes YOLO-format files directly into the bind-mounted `data/labels/` folder. **Open it now in a new tab:**

> **▶ <http://localhost:8000/annotator>**

(Or click the *Open annotator* button on the launcher home page.)

### What you should see

Three panels: a list of every image on the left, a canvas with the current image in the middle, and a class picker plus a live view of the YOLO label file on the right. The keyboard shortcuts at the bottom of the canvas — click-and-drag to draw, number keys to pick a class, arrow keys to move between images — are the fastest way to work. Use the **"jump to next unlabelled"** button (or its keyboard shortcut) to pick up exactly where you left off across sessions — at this scale, scrolling the full image list by hand every time you come back is not a good use of your week.

**Zoom and pan.** Several of these photos contain animals that are small or distant relative to the frame — real wildlife photography, unlike a studio shot, rarely puts the subject front-and-centre at a convenient size. Use the **mouse wheel** to zoom in toward the cursor (up to 12×), and **Shift + click-drag** to pan around once you're zoomed in. The `+` / `-` / `0` keys also work for zoom in / out / fit-to-pane. **Crucially: the saved coordinates are unaffected by zoom level** — zoom is just a viewing aid, the YOLO `.txt` file always contains normalised `[0, 1]` values regardless of how you drew the boxes.

### Your task

**Annotate every image in the pool.** For each one:

1. Look at the image. Identify every animal that's clearly visible.
2. Pick the right class (1–4 keys or the buttons).
3. Draw a tight box around the animal — including everything that's obviously part of it (tail, legs, ears, horns) — excluding background, *not* including parts of other animals.
4. If the animal is **partly cut off by the edge of the image**, still draw a box around the visible part. Truncation is normal in real photography and detectors handle it fine.
5. Some images are **herd shots** with many animals of the same species — draw one box per individual, even when this means a dozen boxes in one photo. This is tedious by design: real annotation work often is, and learning to stay consistent on box 12 the same way you were on box 1 — and box 1,200 — is the actual skill.
6. If you're **unsure of the species**, skip it for now. Better an unlabelled example than a wrong one. (We return to this in Exercise 1.)

The annotator auto-saves on every change. There's no save button. As you draw, watch the *YOLO label file* panel on the right — you're literally writing the rows of the label file as you click and drag.

**Aim for tight boxes.** Loose boxes leak background pixels into the model's idea of what 'elephant' looks like, and the detector learns that 'elephant = lots of savanna with a smaller elephant-shape in it'. Tight is better.

**Pace yourself across the week.** Annotation quality drops sharply once fatigue sets in — a rushed box at hour four of a single session is worse than no box at all. Work in sessions of an hour or two, several times over the week, rather than trying to power through the whole pool in one sitting. The image list panel shows how many boxes each image has, so you can always see exactly where you stopped. Aim for **every image labelled**, with a total somewhere in the range of **2,500-2,800 boxes** across the whole pool — herd shots pull that average well above one box per image.

## 3. Inspect your work — load the labels back into Python

**Always validate your annotations visually.** Reading the raw `.txt` numbers tells you nothing about whether the boxes are sensible. Re-rendering them on top of the images closes the loop and catches:

- Off-by-one errors (e.g. confused row/column order)
- Class-id mistakes ("this is labelled `2 rhino` but actually it's a buffalo")
- Boxes that drifted from the object during a resize
- Truncation issues at image edges

We use the same `PIL` + `matplotlib` we've been using all module. No special libraries needed.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

DATA_DIR = Path("data")
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

n_pool_images = len(list(IMAGES_DIR.glob("*.jpg")))
if n_pool_images == 0:
    raise FileNotFoundError(
        "No images found in data/images/. This pool isn't bundled with the "
        "repo (see data/images/README.md) — add the .jpg files your "
        "instructor provided before continuing."
    )
print(f"Image pool: {n_pool_images} photos found in {IMAGES_DIR}")

# Read the class list — single source of truth.
with open(DATA_DIR / "classes.txt") as f:
    classes = [line.strip() for line in f if line.strip()]
print(f"Classes ({len(classes)}): {classes}")

# A per-class colour list for plotting — used consistently in every later lab too.
CLASS_COLOURS = {
    0: "#d6336c",   # buffalo
    1: "#f59f00",   # elephant
    2: "#2b8a3e",   # rhino
    3: "#1c7ed6",   # zebra
}

In [ ]:
def read_yolo_labels(label_path: Path) -> list[dict]:
    """Parse a YOLO .txt file. Returns a list of dicts with class_id, cx, cy, w, h."""
    if not label_path.is_file():
        return []
    boxes = []
    for line in label_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            print(f"  WARNING: malformed line in {label_path.name}: {line!r}")
            continue
        cls_id = int(parts[0])
        cx, cy, w, h = [float(x) for x in parts[1:]]
        boxes.append({"class_id": cls_id, "cx": cx, "cy": cy, "w": w, "h": h})
    return boxes


def draw_image_with_boxes(image_path: Path, boxes: list[dict], ax=None):
    """Render an image with overlaid bounding boxes on the given axis."""
    img = Image.open(image_path)
    W, H = img.size
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")

    for b in boxes:
        # Convert centre-based normalised back to corner-based pixels.
        x = (b["cx"] - b["w"] / 2) * W
        y = (b["cy"] - b["h"] / 2) * H
        w = b["w"] * W
        h = b["h"] * H
        colour = CLASS_COLOURS.get(b["class_id"], "#888888")
        rect = patches.Rectangle((x, y), w, h, linewidth=2.5,
                                  edgecolor=colour, facecolor="none")
        ax.add_patch(rect)
        # Label background and text
        label = classes[b["class_id"]] if 0 <= b["class_id"] < len(classes) else f"cls{b['class_id']}"
        ax.text(x + 4, y - 6, label, color="white", fontsize=11, fontweight="bold",
                bbox=dict(boxstyle="square,pad=0.2", facecolor=colour, edgecolor="none"))
    return ax


def to_xyxy(b: dict, W: int, H: int) -> list[float]:
    """Convert a centre-based normalised box dict to pixel-space [x1, y1, x2, y2]."""
    return [(b["cx"] - b["w"] / 2) * W, (b["cy"] - b["h"] / 2) * H,
            (b["cx"] + b["w"] / 2) * W, (b["cy"] + b["h"] / 2) * H]


def iou_xyxy(a: list[float], b: list[float]) -> float:
    """IoU of two [x1, y1, x2, y2] boxes. Used in Exercise 2 (inter-annotator agreement)."""
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter == 0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)

Now look at every image with its current annotations. **If you haven't annotated yet, this will show blank images** — open the annotator and label them first.

In [ ]:
# At this scale, plotting every image in one grid isn't practical (1,500+
# subplots). We report aggregate stats over the whole pool, then render a
# random spot-check sample — re-run this cell to see a different sample,
# or use the annotator itself to review the full set.
SAMPLE_SIZE = 16

image_files = sorted(IMAGES_DIR.glob("*.jpg"))

total_boxes = 0
n_labelled = 0
for img_path in image_files:
    boxes = read_yolo_labels(LABELS_DIR / (img_path.stem + ".txt"))
    total_boxes += len(boxes)
    if boxes:
        n_labelled += 1

print(f"Labelled: {n_labelled} / {len(image_files)} images, {total_boxes} boxes total.")

sample = random.sample(image_files, k=min(SAMPLE_SIZE, len(image_files)))
n_cols = 4
n_rows = (len(sample) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.2 * n_rows))
axes = axes.flatten()

for ax, img_path in zip(axes, sample):
    boxes = read_yolo_labels(LABELS_DIR / (img_path.stem + ".txt"))
    draw_image_with_boxes(img_path, boxes, ax=ax)
    ax.set_title(f"{img_path.name}  ({len(boxes)} box{'es' if len(boxes) != 1 else ''})", fontsize=9)

for ax in axes[len(sample):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

**What to look for (in the sample above, and as you spot-check more via the annotator):**

- Every animal in every image has a box, including every individual in the herd shots.
- Every box is tight around its animal (no big gaps of background inside the box).
- Every box has the right class — colour-check it against the species you can see.
- Boxes that look 'shifted' from their animal usually mean an off-by-one bug in the annotator. We've tested ours but if you see something weird, report it.

**Re-open the annotator and fix anything you spot.** The label files update live; re-run the cell above (it re-samples each time) to spot-check different images as you go.

## 4. The `data.yaml` file

`ultralytics` (the library that ships YOLOv8/v11/26) needs one extra file: a `data.yaml` that tells it the dataset layout. It's a small file but it's the contract between your annotations and the trainer.

Required fields:

```yaml
path: /workspace/labs/lab06_image_annotation/data    # absolute path to the dataset root
train: images/train                                    # train images, relative to `path`
val:   images/val                                      # validation images, relative to `path`
test:  images/test                                     # test images (optional but recommended)

names:
  0: buffalo
  1: elephant
  2: rhino
  3: zebra
```

`ultralytics` looks for label files in the *parallel* folder `labels/<split>/` next to each `images/<split>/`. So our flat image pool needs to be reorganised slightly to get there.

## 5. Train / val / test split for object detection

We split into 70% train / 15% val / 15% test, but for object detection this needs a slightly different mindset from classification.

**Important: avoid frame-level leakage.** If your images came from a video — say, three frames at 0.04s intervals showing the same elephant in nearly the same pose — random shuffling will put one frame in train and another in val, and the validation score will be massively optimistic. The model will essentially have seen the validation image during training.

Our images are independent photographs, not video frames, so this isn't a risk here. For *your own* dataset later, **split at the level of unique scenes**, not individual frames.

**A note for next time you inherit a dataset.** Here, you built this split yourself, from scratch — that's correct, because you also built the dataset. But most real-world datasets you'll work with arrive **already split** by whoever published them. In that case, don't be tempted to merge everything back into one pool and re-split it your own way: you'd lose comparability with anyone else's published results on that split, and you risk silently reintroducing exactly the kind of leakage described above if the original authors deliberately kept related images together. The rule of thumb: build your own split only when you built the dataset; inherit the split when you inherited the dataset.

In [ ]:
import shutil
import yaml
import random

RNG_SEED = 7144
SPLIT = (0.7, 0.15, 0.15)

# Find every image that has at least one annotation. Images without
# annotations are excluded from training — you wouldn't want to teach
# the model 'this image is correctly labelled as containing nothing'
# unless you really do want negative examples.
labelled = [p for p in sorted(IMAGES_DIR.glob("*.jpg"))
            if (LABELS_DIR / (p.stem + ".txt")).is_file()
            and read_yolo_labels(LABELS_DIR / (p.stem + ".txt"))]

print(f"Labelled images available: {len(labelled)} of {len(image_files)}")

if len(labelled) < len(image_files):
    print("You haven't annotated every image yet — go back to step 2.")
else:
    rng = random.Random(RNG_SEED)
    rng.shuffle(labelled)
    n = len(labelled)
    n_train = int(n * SPLIT[0])
    n_val = int(n * SPLIT[1])
    splits = {
        "train": labelled[:n_train],
        "val":   labelled[n_train:n_train + n_val],
        "test":  labelled[n_train + n_val:],
    }
    for split_name, files in splits.items():
        print(f"  {split_name:<5}: {len(files)} images")

In [ ]:
# Build the directory layout that ultralytics expects.
# We create images/{train,val,test} and labels/{train,val,test} side-by-side.

OUT_ROOT = DATA_DIR  # we write the split structure right next to the flat one

for split in ("train", "val", "test"):
    (OUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# We *copy* (not move) so the flat data/images/ stays intact for re-annotation.
for split_name, files in splits.items():
    for fp in files:
        shutil.copy2(fp, OUT_ROOT / "images" / split_name / fp.name)
        label_src = LABELS_DIR / (fp.stem + ".txt")
        shutil.copy2(label_src, OUT_ROOT / "labels" / split_name / (fp.stem + ".txt"))
    print(f"  {split_name}: {len(files)} files copied to images/{split_name}/ and labels/{split_name}/")

# Write the data.yaml. We use a relative `path` so the file works regardless
# of where the container is mounted on the host.
data_yaml = {
    "path": str((OUT_ROOT).resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(classes)},
}

yaml_path = OUT_ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"\nWrote {yaml_path}")
print("\nContents:")
print(yaml_path.read_text())

**Sanity check.** When we get to the YOLO training lab, the trainer will fail with a useful error if anything is wrong with this setup. But let's catch problems now while we still know what we did. Three checks:

1. Every image in `images/<split>/` has a corresponding `labels/<split>/<stem>.txt`.
2. Every class id in every label file is within range `0..len(classes)-1`.
3. Every box's coordinates are within `[0, 1]`.

In [ ]:
issues = []

for split in ("train", "val", "test"):
    img_dir = OUT_ROOT / "images" / split
    lbl_dir = OUT_ROOT / "labels" / split
    for img_path in sorted(img_dir.glob("*.jpg")):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.is_file():
            issues.append(f"{split}: no label file for {img_path.name}")
            continue
        for i, line in enumerate(lbl_path.read_text().splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                issues.append(f"{split}/{lbl_path.name} line {i}: not 5 fields")
                continue
            try:
                cls_id = int(parts[0])
                cx, cy, w, h = [float(x) for x in parts[1:]]
            except ValueError:
                issues.append(f"{split}/{lbl_path.name} line {i}: parse error")
                continue
            if not (0 <= cls_id < len(classes)):
                issues.append(f"{split}/{lbl_path.name} line {i}: class_id {cls_id} out of range")
            for fname, v in [("cx", cx), ("cy", cy), ("w", w), ("h", h)]:
                if not (0 <= v <= 1):
                    issues.append(f"{split}/{lbl_path.name} line {i}: {fname}={v} outside [0,1]")

if issues:
    print(f"{len(issues)} issue(s) found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("All checks passed. Your dataset is ready to train on.")

## 6. Exercise 1 — annotation hygiene & edge cases

Real annotation is full of judgement calls. For **each** of the scenarios below, write **2–4 sentences** describing how you would handle it. There isn't always a single right answer — you're being assessed on whether your judgement is **defensible and consistent**.

**(a) Truncation.** An elephant in a herd shot is half-hidden behind another elephant, with only its head and one ear visible. Do you draw a box, and if so, around what — only the visible part, or your best guess of the whole animal's outline?

**(b) Multiple instances stacked.** A herd of eight zebras is bunched tightly together, several overlapping. Do you draw eight separate boxes (one per animal), or fewer, larger boxes around the group?

**(c) Ambiguous species at a distance.** A large grey shape in the far background of a savanna photo could plausibly be a rhino or an elephant at that range and angle. What do you do?

**(d) Off-class object.** Your photo prominently features a giraffe. Giraffes are not in your four-class list. What do you do with it?

**(e) Cropping vs. labelling.** A very out-of-focus animal in the corner of the image — barely identifiable as anything. Annotate or skip?

**(f) Inter-annotator consistency.** You're working with three other students on the same dataset. How do you ensure you're all drawing boxes the same way?

*Your answers:*

**(a) Truncation:**

**(b) Multiple instances stacked:**

**(c) Ambiguous species at a distance:**

**(d) Off-class object:**

**(e) Out-of-focus tiny object:**

**(f) Inter-annotator consistency:**

## 7. Exercise 2 — inter-annotator agreement

**Pair up with one classmate.** Pick **the same single image** from your annotated pool, and re-annotate it independently — neither of you should see the other's boxes while drawing.

Once you've both finished:

1. Compare your two `.txt` files side-by-side.
2. For each box in your file, find the 'matching' box in your partner's file (the one closest in centre and size).
3. For each pair, compute the **Intersection over Union (IoU)** — reuse `iou_xyxy` from Section 3.
4. Report the mean IoU across all matched pairs.
5. If there are 'extra' boxes in one annotator's file that don't appear in the other's, discuss why.

**Discussion.** A mean IoU of 0.9 across annotators is excellent. 0.7 is good. Below 0.5 means you're drawing meaningfully different boxes and you have a *calibration problem* — write one sentence describing what calibration step you'd add to your team's process.

In [ ]:
# Your code for Exercise 2 — comparing your boxes against your partner's.

*Your written reflection on inter-annotator agreement:*

**Mean IoU achieved:**

**Disagreements:**

**Calibration step you'd add:**

---

## 8. Reflection questions

**Q1.** State, in one sentence, what makes object detection a fundamentally harder problem than image classification.

**Q2.** Why does the YOLO label format use **normalised** coordinates (in `[0, 1]`) rather than absolute pixel coordinates? Give one practical benefit and one practical drawback.

**Q3.** The lab earlier insisted that empty label files (for images containing no annotated objects) should *still exist* on disk. What would go wrong if we just left such images without `.txt` files at all?

**Q4.** Section 5 warned against reshuffling a dataset's split once you've inherited it already split by someone else. Give a specific example of a dataset structure (real or hypothetical) where naively shuffling by image would create frame-level leakage, and describe how you'd fix it.

**Q5.** Annotation is expensive — the rule of thumb in industry is roughly 30 seconds per box for trained annotators. Suppose your team has one week and £5,000 of budget and you need a dataset of 5,000 wildlife images annotated for a ten-class list. Outline a plan: do you do it in-house, outsource, use semi-automated tools, or some combination? Justify your choice in 3–5 sentences.

*Your answers:*

**A1.**

**A2.**

**A3.**

**A4.**

**A5.**

---

## What's next

You now have a dataset you built and can vouch for personally — every image in it, annotated by you, at a scale that starts to resemble real annotation work rather than a toy exercise. **In Lab 7 we train an object detector on it** — using a pre-trained YOLO26 model and the Ultralytics library — and see for the first time what a detector trained at a realistic scale can actually do.

Before leaving this lab, make sure:

- [ ] Every image in `data/images/` has at least one bounding box (or was deliberately left unlabelled per Exercise 1(c)/(e))
- [ ] You have completed Exercises 1 and 2
- [ ] You have answered the reflection questions
- [ ] The validation cell in Section 5 reports zero issues
- [ ] `data/data.yaml` exists and has been created with the correct paths
- [ ] Your notebook runs **top to bottom without errors** (*Kernel → Restart and Run All*)